In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from lib.data_quality import data_quality_report, apply_data_quality_rules
from lib.feature_selection import classification_woe_iv, WoETransformer
from lib.feature_engineering import select_columns
from lib.model_selection import compare_models, coefficient_checks
from lib.probability_of_default import ProbabilityOfDefault, evaluate_pd, binomial_calibration_test, psi
from lib.scorecard import pd_to_score

# Load Data and check its quality

In [ ]:
df = pd.read_csv("data/credit_risk_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe(include="all")

In [ ]:
data_quality_report(df)

The data has 165 duplicated rows, which will cause overoptimistic test results after splitting the data, and has some data quality issues (impossible/unlikely employment length, and high age).

The quality rules below set outliers to null as they are data quality issues and the model can account for missingness through WoE transformation (which is shown to be MNAR for employment length below), and removes duplicated rows.

In [ ]:
df.drop_duplicates(inplace=True)
df = apply_data_quality_rules(df)
data_quality_report(df)

In [ ]:
df_mnar = df.copy()

In [ ]:
df_mnar["missing employment length?"] = np.where(df_mnar["person_emp_length"].isnull(), "yes", "no")

df_mnar.groupby("missing employment length?")["loan_status"].agg(["count", "mean"])

Split the data into train and test sets for evaluation

In [ ]:
target_col = "loan_status"

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [ ]:
print(
    "      |   X   | y\n"
    "––––––––––––––––––––––––\n"
    f"Train | {len(X_train)} | {len(y_train)}\n"
    f"Test  | {len(X_test)}  | {len(y_test)}\n"
)

In [ ]:
fig, ax = plt.subplots()

(y_train.value_counts(normalize=True)*100).plot(kind="bar", title="Default distribution in training set", ax=ax)

ax.set_xlabel("Loan Status (Default=1)")
ax.set_ylabel("Percent (%)")

plt.show()

# Feature selection and engineering

In [ ]:
iv, woe = classification_woe_iv(pd.concat([X_train, y_train], axis=1), target_col, show_woe=False)
woe

The below features are selected because they have an information value (IV) that suggests that they are good predictors of default. Note that loan_grade (IV=0.87) and loan_int_rate (IV=0.66) are not included as features in the modelling. This is because these features are circular and also are part of separate modelling within the same process as PD.

In [ ]:
core_feats = ["person_home_ownership", "person_emp_length", "loan_intent", "cb_person_default_on_file"]

In [ ]:
def build_pipeline(feats, custom_bins=None, class_weight=None):
    return Pipeline([
        ("select", FunctionTransformer(select_columns, kw_args={"columns": feats})),
        ("woe",    WoETransformer(variables=feats, custom_bins=custom_bins)),
        ("model",  LogisticRegression(class_weight=class_weight, random_state=42)),
    ])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0)

**Experiment 1:** How does dropping collinear features affect the model?

In [ ]:
affordability_options = {
    "all three":                ["loan_amnt", "person_income", "loan_percent_income"],
    "drop loan_amnt":           ["person_income", "loan_percent_income"],
    "drop person_income":       ["loan_amnt", "loan_percent_income"],
    "drop loan_percent_income": ["loan_amnt", "person_income"],
}
models = {name: build_pipeline(core_feats + extra_feats) for name, extra_feats in affordability_options.items()}

compare_models(models, X_train, y_train, cv, baseline="drop loan_amnt")

In [ ]:
pd.DataFrame({
    name: {"All signs OK": checks["Sign OK"].all(), "Max VIF": checks["VIF"].max()}
    for name, checks in ((n, coefficient_checks(m, X_train, y_train)) for n, m in models.items())
}).T

In [ ]:
# View the coarse WoE values
t = WoETransformer(variables=["loan_amnt"]).fit(X_train, y_train)
print(t.woe_tables_["loan_amnt"][["Cutoff", "N", "WoE"]].round(3))

Keeping all three affordability variables has the best Gini, all of the coefficient signs are valid (negative because all are WoE transformed), and the VIF is good (1.6). The fine-binned loan_amnt WoE is non-monotonic, so Experiment 2 explores coarse classing.

**Experiment 2:** Does coarse classing and WoE transformation of loan amount affect the model, and can the WoE values be monotonic through the bins?

In [ ]:
fine = WoETransformer(variables=["loan_amnt"]).fit(X_train, y_train)
fine.woe_tables_["loan_amnt"][["Cutoff", "N", "Bads", "WoE"]]

In [ ]:
# Compare the candidate groupings
affordability_feats = ["person_income", "loan_percent_income"]
coarse_candidates = {
    "[5000, 12000]":         [5000, 12000],
    "[4400, 10000, 14500]":  [4400, 10000, 14500],
    "[10000, 15000]":        [10000, 15000],
    "[12000]":               [12000],
}

models = {"drop loan_amnt":  build_pipeline(core_feats + affordability_feats),
          "loan_amnt fine":  build_pipeline(core_feats + affordability_feats + ["loan_amnt"])}
for name, cuts in coarse_candidates.items():
    models[f"loan_amnt {name}"] = build_pipeline(core_feats + affordability_feats + ["loan_amnt"],
                                                 custom_bins={"loan_amnt": cuts})

compare_models(models, X_train, y_train, cv, baseline="drop loan_amnt")

In [ ]:
# View the coarse WoE values for each candidate
for name, cuts in coarse_candidates.items():
    t = WoETransformer(variables=["loan_amnt"], custom_bins={"loan_amnt": cuts}).fit(X_train, y_train)
    print(name, t.woe_tables_["loan_amnt"][["Cutoff", "N", "WoE"]].round(3).values.tolist())

In [ ]:
# Stability check: is the hump pattern real or noise?
cuts = [4400, 10000, 14500]
fold_woe = []
for i, (tr, _) in enumerate(StratifiedKFold(5, shuffle=True, random_state=1).split(X_train, y_train)):
    t = WoETransformer(variables=["loan_amnt"], custom_bins={"loan_amnt": cuts})
    t.fit(X_train.iloc[tr], y_train.iloc[tr])
    fold_woe.append(t.woe_tables_["loan_amnt"].set_index("Cutoff")["WoE"].rename(f"fold {i+1}"))

pd.concat(fold_woe, axis=1)

- The fine binned WoE transformed loan amount has the best Gini, but it's WoE values are non-monotonic - there is a hump at bins 4/5.
- The coarse bins [5000, 12000] and [4400, 10000, 14500] are the next best Gini models, but their WoE values are also non-monotonic.
- The next best bins are **[10000, 15000]**. This bin does have monotonic WoE values, and it makes business sense.

In [ ]:
feats = core_feats + affordability_feats + ["loan_amnt"]
custom_bins = {"loan_amnt": [10000, 15000]}

# Model selection

**Experiment 3:** Check whether class weight balancing and calibration affect the model

In [ ]:
calibration_candidates = {
    "balanced, no calibration":   ProbabilityOfDefault(build_pipeline(feats, custom_bins, "balanced"), calibration=None),
    "balanced + isotonic":        ProbabilityOfDefault(build_pipeline(feats, custom_bins, "balanced"), calibration="isotonic"),
    "unweighted, no calibration": ProbabilityOfDefault(build_pipeline(feats, custom_bins), calibration=None),
    "unweighted + isotonic":      ProbabilityOfDefault(build_pipeline(feats, custom_bins), calibration="isotonic"),
    "unweighted + sigmoid":       ProbabilityOfDefault(build_pipeline(feats, custom_bins), calibration="sigmoid"),
}

results, oof = {}, {}
for name, candidate in calibration_candidates.items():
    p = candidate.out_of_fold_pd(X_train, y_train)
    oof[name] = p
    low = p <= np.quantile(p, 0.1)
    results[name] = pd.concat([evaluate_pd(y_train, p),
                               pd.Series({"Low decile pred (%)": p[low].mean() * 100,
                                          "Low decile obs (%)":  y_train[low].mean() * 100})])

pd.DataFrame(results).T.round(4)

In [ ]:
binomial_calibration_test(y_train.values, oof["unweighted + isotonic"], strategy="quantile")

In [ ]:
b = build_pipeline(feats, custom_bins, "balanced").fit(X_train, y_train)[-1]
u = build_pipeline(feats, custom_bins).fit(X_train, y_train)[-1]
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Intercept shift: {b.intercept_[0] - u.intercept_[0]:.3f}  vs  ln(ratio) = {np.log(ratio):.3f}")

Gini is effectively the same across all options, but unweighted + isotonic calibration wins in most other fields (Gini=0.6319, Brier=0.1186, Log-loss=0.3869, HL=8.4412, HL_p=0.3916). The binomial goodness-of-fit test for the unweighted + isotonic calibration model (out-of-fold) also looks good, particularly in the lowest-risk bin (although slight under-prediction).

## Final model

In [ ]:
pipe = build_pipeline(feats, custom_bins=custom_bins) # no balancing class weights
pd_model = ProbabilityOfDefault(pipe).fit(X_train, y_train)

# Probability of Default Modelling and Evaluation

## Cross-validation

In [ ]:
oof_pd = pd_model.out_of_fold_pd(X_train, y_train)
evaluate_pd(y_train, oof_pd)

In [ ]:
binomial_calibration_test(y_train, oof_pd)

## Train vs test performance

In [ ]:
pd.DataFrame({"Train": pd_model.evaluate(X_train, y_train),
              "Test":  pd_model.evaluate(X_test, y_test)}).round(4)

In-sample calibration (train HL p = 0.97) is optimistic, because the calibrator was fitted on the same data; out-of-fold figures are the reliable training-data estimate.

## Model calibration on test set

In [ ]:
calib = pd_model.calibration_table(X_test, y_test)

fig, ax = plt.subplots()
ax.plot([0, 1], [0, 1], color="black", linestyle="--")
calib.plot(x="expected_prob", y="observed_prob", marker="o", title="Calibration (test)", ax=ax)
plt.show()

calib

## Stability across splits

In [ ]:
rows = []
for seed in range(20):
    a, b, c, d = train_test_split(X, y, stratify=y, test_size=0.2, random_state=seed)
    rows.append(ProbabilityOfDefault(pipe).fit(a, c).evaluate(b, d).rename(seed))

splits = pd.DataFrame(rows)
print(splits[["Gini", "HL", "HL p"]].describe().loc[["mean", "50%", "min", "max"]].round(3))
print(f"Splits failing HL at 5%: {(splits['HL p'] < 0.05).mean():.0%}")

## Population Stability Index

In [ ]:
psi_table = psi(pd_model.predict_pd(X_train), pd_model.predict_pd(X_test))
print(f"Total PSI: {psi_table['PSI'].sum():.4f}")
psi_table

## Scores

In [ ]:
test_pd = pd_model.predict_pd(X_test)
pd.DataFrame({"Actual": y_test,
              "PD (%)": (test_pd * 100).round(1),
              "Score":  pd_to_score(test_pd)}).head(10)

# Findings and Limitations

Out-of-fold calibration is good (HL p = 0.39), but on held-out splits HL fails in 25% of cases (vs about 5% expected), with under-prediction concentrated in the lowest-PD bin. This is consistent with isotonic calibration's weakness at the extremes. Possible remedies: smoother calibration, a minimum-events constraint for the lowest bin, or a margin of conservatism on low PDs.